# IBM AML Demo Queries

This notebook runs Gremlin queries and shows query results.
Start backend first in another terminal:
`mvn exec:java`


In [ ]:
import requests
import pandas as pd
import subprocess
import json as _json
from typing import Dict, Any
from IPython.display import display, Markdown
from pathlib import Path

BASE_URL = "http://localhost:7000"
MAX_ROWS = 100000
REPO_ROOT = Path.home() / "SourceCode/graph-query-engine"
SHOW_PLAN = False


def run_aml_data_download(variant: str = "HI-Small", rows: int = 100000) -> bool:
    # Download HI-Small files and generate demo/data/aml-demo.csv via normalize_aml.py.
    if not REPO_ROOT.exists():
        display(Markdown(f"**Error:** repo path not found: `{REPO_ROOT}`"))
        return False

    cmd = f"bash ./scripts/download_aml_data.sh --variant {variant} --rows {rows}"
    display(Markdown(f"Running: `{cmd}` in `{REPO_ROOT}`"))
    proc = subprocess.run(cmd, cwd=REPO_ROOT, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc.returncode == 0


def resolve_raw_source_csv() -> str | None:
    candidates = [
        Path.cwd() / "demo/data/HI-Small_Trans.csv",
        Path.cwd() / "data/HI-Small_Trans.csv",
    ]
    for p in candidates:
        if p.exists():
            return str(p)

    demo_data = Path.cwd() / "demo/data"
    if demo_data.exists():
        matches = sorted(demo_data.glob("HI-*_Trans.csv"))
        if matches:
            return str(matches[0])
    return None


def normalized_csv_path() -> str:
    return str(Path.cwd() / "demo/data/aml-demo.csv")


def _count_rows_upto(csv_path: str, max_rows: int) -> int:
    import csv
    rows = 0
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)
        for _ in reader:
            rows += 1
            if rows >= max_rows:
                break
    return rows


def normalize_for_max_rows(src_csv: str, max_rows: int) -> str | None:
    out_csv = normalized_csv_path()
    normalize_py = REPO_ROOT / "scripts/normalize_aml.py"
    if not normalize_py.exists():
        display(Markdown(f"**normalize_aml.py missing:** `{normalize_py}`"))
        return None

    cmd = [
        "python3",
        str(normalize_py),
        "--src",
        str(src_csv),
        "--dst",
        str(out_csv),
        "--rows",
        str(max_rows),
    ]
    display(Markdown(f"Preparing normalized CSV for MAX_ROWS via: `{ ' '.join(cmd) }`"))
    proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        return None
    return out_csv if Path(out_csv).exists() else None


def resolve_mapping_path() -> str | None:
    candidates = [
        Path.cwd() / "mappings/aml-mapping.json",
        Path.cwd() / "demo/mappings/aml-mapping.json",
    ]
    for p in candidates:
        if p.exists():
            return str(p)
    return None


CSV_PATH = normalized_csv_path() if Path(normalized_csv_path()).exists() else None
RAW_SOURCE_CSV = resolve_raw_source_csv()
MAPPING_PATH = resolve_mapping_path()
print("BASE_URL:", BASE_URL)
print("CSV_PATH:", CSV_PATH)
print("RAW_SOURCE_CSV:", RAW_SOURCE_CSV)
print("MAPPING_PATH:", MAPPING_PATH)
print("MAX_ROWS:", MAX_ROWS)


def try_upload_mapping() -> bool:
    if not MAPPING_PATH:
        display(Markdown("**AML mapping not found.** Expected `mappings/aml-mapping.json` or `demo/mappings/aml-mapping.json`."))
        return False

    try:
        with open(MAPPING_PATH, "rb") as mapping_file:
            mapping_resp = requests.post(
                f"{BASE_URL}/mapping/upload",
                files={"file": mapping_file},
                timeout=30,
            )
    except Exception as e:
        display(Markdown(f"**Mapping upload failed:** {e}"))
        return False

    if mapping_resp.status_code not in (200, 201):
        display(Markdown(f"**Mapping upload failed** (`HTTP {mapping_resp.status_code}`):"))
        print(mapping_resp.text)
        return False

    print(mapping_resp.json())
    return True


def get_account_count() -> int | None:
    try:
        count_resp = requests.post(
            f"{BASE_URL}/gremlin/query",
            json={"gremlin": "g.V().hasLabel('Account').count()"},
            timeout=30,
        )
        if count_resp.status_code != 200:
            display(Markdown(f"**Data verification failed** (`HTTP {count_resp.status_code}`):"))
            print(count_resp.text)
            return None

        payload = count_resp.json()
        values = payload.get("results") or []
        if not values:
            return 0
        return int(values[0])
    except Exception as e:
        display(Markdown(f"**Data verification error:** {e}"))
        return None


def get_transfer_count() -> int | None:
    try:
        count_resp = requests.post(
            f"{BASE_URL}/gremlin/query",
            json={"gremlin": "g.E().hasLabel('TRANSFER').count()"},
            timeout=30,
        )
        if count_resp.status_code != 200:
            display(Markdown(f"**Transfer count check failed** (`HTTP {count_resp.status_code}`):"))
            print(count_resp.text)
            return None

        payload = count_resp.json()
        values = payload.get("results") or []
        if not values:
            return 0
        return int(values[0])
    except Exception as e:
        display(Markdown(f"**Transfer count check error:** {e}"))
        return None


def count_csv_rows(csv_path: str, max_rows: int) -> int:
    return _count_rows_upto(csv_path, max_rows)


def run_aml_loader_script(csv_path: str, max_rows: int) -> bool:
    if not MAPPING_PATH:
        display(Markdown("**AML mapping not found.** Expected `mappings/aml-mapping.json` or `demo/mappings/aml-mapping.json`."))
        return False

    if not REPO_ROOT.exists():
        display(Markdown(f"**Error:** repo path not found: `{REPO_ROOT}`"))
        return False

    cmd = [
        "python3",
        "demo/aml_csv_loader.py",
        "--path",
        str(csv_path),
        "--max-rows",
        str(max_rows),
        "--url",
        BASE_URL,
        "--mapping",
        str(MAPPING_PATH),
    ]
    display(Markdown(f"Running loader script: `{ ' '.join(cmd) }`"))
    proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc.returncode == 0


def ensure_csv_present(auto_download: bool = True, max_rows: int = 100_000) -> str | None:
    global CSV_PATH, RAW_SOURCE_CSV

    if CSV_PATH and Path(CSV_PATH).exists():
        existing = _count_rows_upto(CSV_PATH, max_rows)
        if existing >= max_rows:
            return CSV_PATH

    if not RAW_SOURCE_CSV:
        RAW_SOURCE_CSV = resolve_raw_source_csv()

    if not RAW_SOURCE_CSV and auto_download:
        ok = run_aml_data_download(variant="HI-Small", rows=max_rows)
        if ok:
            RAW_SOURCE_CSV = resolve_raw_source_csv()

    if not RAW_SOURCE_CSV:
        return None

    prepared = normalize_for_max_rows(RAW_SOURCE_CSV, max_rows)
    if prepared:
        CSV_PATH = prepared
    return CSV_PATH


def ensure_aml_ready(auto_download: bool = True, max_rows: int = 100_000) -> None:
    csv_path = ensure_csv_present(auto_download=auto_download, max_rows=max_rows)
    if not csv_path:
        display(Markdown("**CSV not found after download attempt.**"))
        return

    display(Markdown(f"CSV detected: `{csv_path}`"))

    account_count = get_account_count()
    if account_count is None:
        return

    transfer_count = get_transfer_count()
    if transfer_count is None:
        return

    expected_rows = count_csv_rows(csv_path, max_rows)
    display(Markdown(f"CSV rows considered (up to MAX_ROWS): **{expected_rows}**"))
    display(Markdown(f"Current graph TRANSFER edges: **{transfer_count}**"))

    if transfer_count == expected_rows and expected_rows > 0:
        # Keep mapping in sync for query/explain even when data already exists.
        try_upload_mapping()
        display(Markdown(f"AML data already loaded and up-to-date (`TRANSFER={transfer_count}`)."))
        return

    display(Markdown("Dataset not loaded (or row count mismatch); running AML loader script..."))
    loaded = run_aml_loader_script(csv_path, max_rows)
    if not loaded:
        display(Markdown("**Loader script failed.** Check output above."))
        return

    refreshed_accounts = get_account_count()
    refreshed_transfers = get_transfer_count()
    if refreshed_accounts is not None and refreshed_transfers is not None:
        display(Markdown(
            f"AML data load complete: **{refreshed_accounts}** Account vertices, "
            f"**{refreshed_transfers}** TRANSFER edges."
        ))


## Step 1: Prepare CSV (HI-Small)

Step 2 reuses existing `HI-Small_Trans.csv` and normalizes `aml-demo.csv` for the current `MAX_ROWS`.
If raw HI-Small files are missing, run this in terminal:

```zsh
cd ~/SourceCode/graph-query-engine
bash ./scripts/download_aml_data.sh --variant HI-Small --rows 100000
```

This downloads `HI-Small_Trans.csv` and prepares `demo/data/aml-demo.csv`.

You can also run the helper below from notebook.


In [ ]:
AUTO_RUN_DOWNLOAD = False
DOWNLOAD_VARIANT = "HI-Small"
DOWNLOAD_ROWS = 100000

if AUTO_RUN_DOWNLOAD:
    ok = run_aml_data_download(variant=DOWNLOAD_VARIANT, rows=DOWNLOAD_ROWS)
    if ok:
        RAW_SOURCE_CSV = resolve_raw_source_csv()
        CSV_PATH = normalize_for_max_rows(RAW_SOURCE_CSV, DOWNLOAD_ROWS) if RAW_SOURCE_CSV else None
        print("CSV_PATH refreshed:", CSV_PATH)
else:
    print("Set AUTO_RUN_DOWNLOAD=True to run download + normalize from notebook.")


In [ ]:
def get_sql_explain(gremlin_query: str, include_plan: bool = False) -> Dict[str, Any]:
    try:
        response = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            params={"plan": "true"} if include_plan else {},
            timeout=10,
        )
        if response.ok:
            return response.json()
        return {"error": f"HTTP {response.status_code}: {response.text}"}
    except Exception as e:
        return {"error": str(e)}


def run_gremlin_query(gremlin_query: str, tx_mode: bool = False) -> Dict[str, Any]:
    endpoint = "/gremlin/query/tx" if tx_mode else "/gremlin/query"
    result: Dict[str, Any] = {}
    try:
        response = requests.post(
            f"{BASE_URL}{endpoint}",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            timeout=60,
        )
        result = response.json()
    except Exception as e:
        result = {"error": str(e)}
    result["_sql_explain"] = get_sql_explain(gremlin_query, include_plan=SHOW_PLAN)
    return result


def display_query_result(gremlin: str, result: Dict[str, Any], title: str = "", limit: int = 10, tx_mode: bool = False):
    if title:
        display(Markdown(f"### {title}"))
    display(Markdown("**Gremlin:**"))
    display(Markdown(f"```groovy\n{gremlin}\n```"))

    explain = result.get("_sql_explain", {})
    if "error" in explain:
        display(Markdown(f"**SQL Translation:** *not available ({explain['error']})*"))
    else:
        sql = explain.get("translatedSql", "")
        params = explain.get("parameters", [])
        plan = explain.get("plan")
        if sql:
            display(Markdown("**SQL Translation:**"))
            display(Markdown(f"```sql\n{sql}\n```"))
            if params:
                display(Markdown(f"**Parameters:** `{params}`"))
        if plan:
            display(Markdown("**Query Plan:**"))
            display(Markdown(f"```json\n{_json.dumps(plan, indent=2)}\n```"))

    if "error" in result:
        display(Markdown(f"**Error:** {result['error']}"))
        return

    rows = result.get("results", [])
    display(Markdown(f"**Result Count:** {result.get('resultCount', 0)}"))
    if not rows:
        return
    if isinstance(rows[0], dict):
        display(pd.DataFrame(rows[:limit]))
    else:
        for i, row in enumerate(rows[:limit], 1):
            print(f"{i}. {row}")


## Step 0: Health check


In [ ]:
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5).text
    provider = requests.get(f"{BASE_URL}/gremlin/provider", timeout=5).json().get("provider", "unknown")
    display(Markdown(f"Status: `{health}`"))
    display(Markdown(f"Provider: `{provider}`"))
except Exception as e:
    display(Markdown(f"Health check failed: {e}"))


## Step 2: Validate CSV, Upload Mapping, Verify Data


In [ ]:
AUTO_DOWNLOAD_IF_MISSING = True
ensure_aml_ready(auto_download=AUTO_DOWNLOAD_IF_MISSING, max_rows=MAX_ROWS)


## Query Sections


## Simple Queries


### S1 Count accounts

Count unique Account vertices.


In [ ]:
gremlin = "g.V().hasLabel('Account').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S1 Count accounts', limit=10, tx_mode=False)


### S2 Count banks

Count Bank vertices - one per distinct bank ID in the data.


In [ ]:
gremlin = "g.V().hasLabel('Bank').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S2 Count banks', limit=10, tx_mode=False)


### S3 Count countries

Count Country vertices (10 pre-seeded jurisdictions).


In [ ]:
gremlin = "g.V().hasLabel('Country').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S3 Count countries', limit=10, tx_mode=False)


### S4 Count alerts

Count Alert vertices - one raised per suspicious transfer.


In [ ]:
gremlin = "g.V().hasLabel('Alert').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S4 Count alerts', limit=10, tx_mode=False)


### S5 Count transfers

Count all TRANSFER edges.


In [ ]:
gremlin = "g.E().hasLabel('TRANSFER').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S5 Count transfers', limit=10, tx_mode=False)


### S6 Suspicious transfer count

Count confirmed suspicious TRANSFER edges (isLaundering=1).


In [ ]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S6 Suspicious transfer count', limit=10, tx_mode=False)


### S7 High-risk countries

List Country vertices with riskLevel=HIGH.


In [ ]:
gremlin = "g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S7 High-risk countries', limit=10, tx_mode=False)


### S8 High-severity alerts

Show HIGH-severity open alerts.


In [ ]:
gremlin = "g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S8 High-severity alerts', limit=15, tx_mode=False)


## Complex Queries


### C1 Top sender accounts

Accounts ranked by outgoing transfer count - find the biggest hubs.


In [ ]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C1 Top sender accounts', limit=15, tx_mode=False)


### C2 Suspicious hubs

Accounts with the most suspicious outgoing transfers.


In [ ]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C2 Suspicious hubs', limit=15, tx_mode=False)


### C3 Account -> Bank (BELONGS_TO)

Show which bank each account belongs to via BELONGS_TO.


In [ ]:
gremlin = "g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C3 Account -> Bank (BELONGS_TO)', limit=15, tx_mode=False)


### C4 Bank -> Country (LOCATED_IN)

Show which country each bank is headquartered in.


In [ ]:
gremlin = "g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C4 Bank -> Country (LOCATED_IN)', limit=15, tx_mode=False)


### C5 Two-hop: Account -> Bank -> Country

Traverse Account->Bank->Country in two hops.


In [ ]:
gremlin = "g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C5 Two-hop: Account -> Bank -> Country', limit=10, tx_mode=False)


### C6 Accounts sending to high-risk countries (SENT_VIA)

Find accounts routing money via FATF-blacklisted countries.


In [ ]:
gremlin = "g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C6 Accounts sending to high-risk countries (SENT_VIA)', limit=20, tx_mode=False)


### C7 Flagged accounts with alert detail (FLAGGED_BY)

Show investigated accounts with linked Alert severity.


In [ ]:
gremlin = "g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C7 Flagged accounts with alert detail (FLAGGED_BY)', limit=20, tx_mode=False)


### C8 Cross-bank suspicious flow

Suspicious transfers that cross bank boundaries.


In [ ]:
gremlin = "g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C8 Cross-bank suspicious flow', limit=15, tx_mode=False)


### C9 Three-hop money trail

Follow suspicious TRANSFER hops 3 steps deep.


In [ ]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C9 Three-hop money trail', limit=10, tx_mode=False)


### C10 Five-hop layering chain

Extended 5-hop traversal for layering detection.


In [ ]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C10 Five-hop layering chain', limit=10, tx_mode=False)


### C11 Transactional suspicious count

Suspicious transfer count via transactional endpoint.


In [ ]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_gremlin_query(gremlin, tx_mode=True)
display_query_result(gremlin, result, title='C11 Transactional suspicious count', limit=10, tx_mode=True)


## Iceberg Note

Use `aml_iceberg_queries.ipynb` for Iceberg comparisons.


## Done
